# TFMN2 Processing

## Overview

**Before sequencing**:
1. ~~From extracted robotic OD data, create table of TransferSamples.~~
2.  ~~From Nidhi and, get table with selected seqsamples. Curate. Verify correctness (can't have same seqsample in two different wells, can't have two samples in one well, all wells must exist - must start at A1 and be filled consecutively). Fix mistakes after discussion if necessary.~~
3.  ~~Merge two tables (seqsamples).~~
4.  ~~Create LIMS seqorder entry (and note if unrelated seqsamples are included), upload seqsamples to LIMS seqsamples table, and point seqsamples to seqorder (Google LIMS)  - delete table~~
5.  ~~create seqsamples plasmidsaurus name list  - delete table~~

**After sequencing**:
1. Create SeqOrder, Libraries, and SeqSamples. Verify seqsamples against LIMS.
2. From seqsamples, make LIMS measurements entries (not all seqsamples may be part of this experiment, so allow for other peoples samples) (all remaining seqsamples are measurements, but not all measurements are seqsamples) - delete table
3. Run analyses on seqsamples and upload results to LIMS, and link to them in measurements table



## Set-Up

In [ ]:
## Make sure running in aisynbio_env

In [1]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

## Sync Google LIMS with DB and get number of expected samples

In [4]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)

%run util_simple.py

✓ LIMS API loaded successfully


In [ ]:
# Run a manual LIMS mirror db sync

from aisynbiopipeline.limsapi.sync import sync_all_sheets

sync_all_sheets()

In [ ]:
# LIMS_samples = query_lims(
#     'robotic_mt_samples',
#     filters={'Experiment': ['TFMN2']}
# )
# n_samples = len(LIMS_samples)

In [5]:
LIMS_wells = query_lims('Wells')
LIMS_wells_byRow = LIMS_wells.sort_values(['Row', 'Column'])['Well']
LIMS_wells_byCol = LIMS_wells.sort_values(['Column', 'Row'])['Well']

## Get TransferSamples

In [7]:
import pandas as pd

paths_to_OD_data = [
    '/storage/synbio/ai_synbio_data/experimental_data/robotic_od_data/TFMN2_01-14-26/TFMN2_robotic_OD_processed_final.csv',
    '/storage/synbio/ai_synbio_data/experimental_data/robotic_od_data/TFMN3_02-04-26/TFMN3_robotic_OD_processed_final.csv'
]
path_to_Nidhi_table = '/storage/nspahr/tmp/TFMN2 and TFMN3 sequencing plate.csv'

OD_df = pd.DataFrame()
for path in paths_to_OD_data:
    df_to_add = pd.read_csv(path).convert_dtypes().rename(columns={'Name':'sample_name'})
    OD_df = pd.concat([OD_df, df_to_add])

Nidhi_all_df = pd.read_csv(path_to_Nidhi_table).convert_dtypes()

expected_cols = ['Sequencing sample',
                 'Experiment',
                 'Sequencing plate',
                 'Sequencing plate well',
                 'Population or Single colony?',
                 'Robotic run plate',
                 'Robotic run plate well']

[print(x) for x in expected_cols if x not in Nidhi_all_df.columns]

expected_exp = ['TFMN2', 'TFMN3']
[print(x) for x in expected_exp if x not in Nidhi_all_df.Experiment.unique()]

Nidhi_df = Nidhi_all_df.loc[Nidhi_all_df['Experiment'].isin(expected_exp)]
Nidhi_df = Nidhi_df.sort_values(['Experiment', 'Sequencing plate','Sequencing plate well'])

other_df = Nidhi_all_df.loc[~Nidhi_all_df['Experiment'].isin(expected_exp)]

In [8]:
OD_df.columns

Index(['filename', 'experiment', 'file_ID', 'timestamp', 'series',
       'plate_index', 'transfer', 'reading', 'row', 'column', 'od', 'well',
       'measurement_type', 'culture_container', 'plate_type', 'start_date',
       'file_basename', 'bmg filename', 'datetime', 'resource id',
       'sample_name', 'Experiment', 'Type', 'Condition', 'strain',
       'Transforming DNA', 'Protocol', 'Parent sample', 'Replicate samples',
       'Plate name', 'Microtiter plate well', 'Unnamed: 11', 'background',
       'innoculation_timestamp', 'timepoint', 'trans_DNA concentration',
       'trans_DNA+conc'],
      dtype='object')

In [9]:
OD_df.head()

,filename,experiment,file_ID,timestamp,series,plate_index,transfer,reading,row,column,...,Parent sample,Replicate samples,Plate name,Microtiter plate well,Unnamed: 11,background,innoculation_timestamp,timepoint,trans_DNA concentration,trans_DNA+conc
0,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,0,...,ANLstock.ACN3575.colony1,"TFMN2.ACN3575.Iva.noDNA.1, TFMN2.ACN3575.Iva.n...",exp1,A1,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>
1,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,1,...,ANL.stock.ACN3500.colony2,"TFMN2.ACN3500.Iva.DEL6kb.1, TFMN2.ACN3500.Iva....",exp1,A2,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>
2,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,2,...,ANL.stock.ACN3500.colony2,"TFMN2.ACN3500.Iva.DEL6kb-DELvanK.1, TFMN2.ACN3...",exp1,A3,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>
3,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,3,...,ANL.stock.ACN3500.colony2,"TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.1, TF...",exp1,A4,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>
4,/Users/nataschaspahr/data/robotic_OD_data/TFMN...,TFMN2,01KEZ2PD3SPDJPH0QQJ4KTSZKV,1768422179,exp1,0,0,T12,0,4,...,ANL.stock.ACN3500.colony2,"TFMN2.ACN3500.Iva.gDNA3560.1, TFMN2.ACN3500.Iv...",exp1,A5,<NA>,0.03875,2026-01-14 20:28:21.089301,0.0,<NA>,<NA>


In [10]:
Nidhi_df

,Experiment,Sequencing plate,Sequencing plate well,Robotic run plate,Robotic run plate well,Population or Single colony?,Sequencing sample
0,TFMN2,1,A1,1,B2,P,<NA>
9,TFMN2,1,A10,1,C2,P,<NA>
10,TFMN2,1,A11,1,C3,P,<NA>
11,TFMN2,1,A12,1,C4,P,<NA>
1,TFMN2,1,A2,1,B3,P,<NA>
...,...,...,...,...,...,...,...
247,TFMN3,3,E8,8,C3,P,<NA>
248,TFMN3,3,E9,8,D3,P,<NA>
252,TFMN3,3,F1,12,D3,P,<NA>
253,TFMN3,3,F2,12,E3,P,<NA>


In [11]:
other_df

,Experiment,Sequencing plate,Sequencing plate well,Robotic run plate,Robotic run plate well,Population or Single colony?,Sequencing sample
255,Gyorgy,3,F4,<NA>,<NA>,P,GB1
256,Gyorgy,3,F5,<NA>,<NA>,P,GB2
257,Gyorgy,3,F6,<NA>,<NA>,P,GB3
258,Gyorgy,3,F7,<NA>,<NA>,P,GB4
259,Gyorgy,3,F8,<NA>,<NA>,P,GB5
260,Gyorgy,3,F9,<NA>,<NA>,P,GB6
261,Gyorgy,3,F10,<NA>,<NA>,P,GB7


In [12]:
# Function to cycle wells list to length

def cycle_list_to_length(original_list, total_elements):
    from itertools import cycle, islice
    
    # Create a cycle iterator
    list_cycle = cycle(original_list)
    
    # Take only the required number of elements using islice
    new_list = list(islice(list_cycle, total_elements))
    
    return new_list

In [13]:
transfersamples = OD_df[['experiment', 'sample_name', 'transfer', 'well']]
transfersamples = transfersamples.loc[(~pd.isna(transfersamples['sample_name'])) & (transfersamples['transfer']!=0)]
transfersamples.drop_duplicates(inplace=True)

In [14]:
print(len(transfersamples.columns) == 4)
print(len(Nidhi_all_df.columns) == 7)

True
True


In [16]:
# Check that

## Selected robotic run plate transfer samples must be present in transfersamples
available = set(map(tuple, transfersamples[['experiment', 'transfer', 'well']].drop_duplicates().values))
required = set(map(tuple, Nidhi_df[['Experiment', 'Robotic run plate', 'Robotic run plate well']].drop_duplicates().values))
if len(required) - len(available)>0:
    print("The following selected robotic run plate transfer samples are missing from the OD_data:")
    for i in required-available:
        print(i)

## seqplates must be consecutive integers
min_seq_plate = min(Nidhi_all_df['Sequencing plate'].unique())
max_seq_plate = max(Nidhi_all_df['Sequencing plate'].unique())
if not all([x in Nidhi_all_df['Sequencing plate'].unique() for x in list(range(min_seq_plate, max_seq_plate+1))]):
    print("The sequencing plate indexes are not consecutive integers:")
    print(sorted(Nidhi_all_df['Sequencing plate'].unique()))

## Seqwells must be in wells
incorrect_wells = set(Nidhi_all_df['Sequencing plate well'].unique()) - set(LIMS_wells_byRow)
if len(incorrect_wells) > 0:
    print("The following selected sequencing plate well locations are incorrect:")
    print(incorrect_wells)

## Seqwells must be in consecutive order
expected_seq_plate_wells = cycle_list_to_length(LIMS_wells_byRow, len(Nidhi_all_df))
if not(Nidhi_all_df['Sequencing plate well'].to_list() == expected_seq_plate_wells):
    print("Sequencing plate wells are not in consecutive order.")
           
## All robotic run plate-well combos are unique
duplicated_roboplate_locs = Nidhi_df[['Experiment', 'Robotic run plate','Robotic run plate well']].duplicated()
if sum(duplicated_roboplate_locs)!=0:
    print("The following robotic run plate-well combos are non-unique:")
    display(Nidhi_df[['Experiment', 'Robotic run plate','Robotic run plate well']][duplicated_roboplate_locs])

## All seqplate-well combos are unique <-- This check would have caught TFMN1 mistake
duplicated_seqplate_locs = Nidhi_all_df[['Sequencing plate','Sequencing plate well']].duplicated()
if sum(duplicated_seqplate_locs)!=0:
    print("The following Sequencing plate-well combos are non-unique:")
    display(Nidhi_all_df[['Sequencing plate','Sequencing plate well']][duplicated_duplicated_seqplate_locs_locs])

In [23]:
seqsamples = pd.merge(transfersamples, Nidhi_df,
                      left_on=['experiment', 'transfer', 'well'],
                      right_on=['Experiment', 'Robotic run plate','Robotic run plate well'],
                      how='right')
seqsamples.drop(['experiment', 'transfer', 'well'], axis=1, inplace=True)

seqsamples.rename(columns={'sample_name': 'Sample Name'}, inplace=True)
seqsamples['Sequencing sample'] = seqsamples['Sample Name'] + '.' + 'T' + seqsamples['Robotic run plate'].astype(str) + '.' + seqsamples['Population or Single colony?']
seqsamples['Robotic run plate well row'] = seqsamples['Robotic run plate well'].apply(lambda x: x[0])
seqsamples['Robotic run plate well column'] = seqsamples['Robotic run plate well'].apply(lambda x: int(x[1:]))

seqsamples = pd.concat([seqsamples, other_df])
seqsamples['Sequencing plate well row'] = seqsamples['Sequencing plate well'].apply(lambda x: x[0])
seqsamples['Sequencing plate well column'] = seqsamples['Sequencing plate well'].apply(lambda x: int(x[1:]))

seqsamples = seqsamples[[
    'Sequencing sample',
    'Experiment',
    'Sample Name',
    'Sequencing plate',
    'Sequencing plate well',
    'Sequencing plate well row',
    'Sequencing plate well column',
    'Population or Single colony?',
    'Robotic run plate',
    'Robotic run plate well',
    'Robotic run plate well row',
    'Robotic run plate well column'
]]

byRow = ['Sequencing plate', 'Sequencing plate well row', 'Sequencing plate well column']
byCol = ['Sequencing plate', 'Sequencing plate well column', 'Sequencing plate well row']
seqsamples.sort_values(byRow, inplace=True)

In [24]:
seqsamples

,Sequencing sample,Experiment,Sample Name,Sequencing plate,Sequencing plate well,Sequencing plate well row,Sequencing plate well column,Population or Single colony?,Robotic run plate,Robotic run plate well,Robotic run plate well row,Robotic run plate well column
0,TFMN2.ACN3500.Iva.DEL6kb.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb.2,1,A1,A,1,P,1,B2,B,2.0
4,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK.2,1,A2,A,2,P,1,B3,B,3.0
5,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.DEL6kb-DELvanK-DELadeK.2,1,A3,A,3,P,1,B4,B,4.0
6,TFMN2.ACN3500.Iva.gDNA3560.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.gDNA3560.2,1,A4,A,4,P,1,B5,B,5.0
7,TFMN2.ACN3500.Iva.gDNA3575.2.T1.P,TFMN2,TFMN2.ACN3500.Iva.gDNA3575.2,1,A5,A,5,P,1,B6,B,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...
257,GB3,Gyorgy,<NA>,3,F6,F,6,P,<NA>,<NA>,NaN,NaN
258,GB4,Gyorgy,<NA>,3,F7,F,7,P,<NA>,<NA>,NaN,NaN
259,GB5,Gyorgy,<NA>,3,F8,F,8,P,<NA>,<NA>,NaN,NaN
260,GB6,Gyorgy,<NA>,3,F9,F,9,P,<NA>,<NA>,NaN,NaN


In [25]:
path_to_seqsamples_table = '/storage/nspahr/RoboticRun_analysis/LIMS_seqsamples.csv'
seqsamples.to_csv(path_to_seqsamples_table, index=False)

## Download from seq facility, file organization, renaming
- Create seqorder name and makedir into reception folder (/synbio/ai_synbio_data/experimental_data/downloads - should this be temporary?). Also make homedir to copy and view analysis reports.
- Download data into folder.
- Spot-check if duplicate read ID problem is fixed.
- Create new seqorder folder in experimental_data/sequencing_data/ and respective libraries.
- Copy all illumina fastqs into short lib, renaming in the process.

In [ ]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_code = 'GFHFWC'
seqorder_name = create_seqorder_name(item_code)

reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
os.makedirs(reception_dir)
print(reception_dir)

home_dir = '/storage/nspahr/lib_analysis/' + seqorder_name
breseq_dir = home_dir + '/breseq_analysis_populations'
os.makedirs(breseq_dir, exist_ok=True)

In [ ]:
# Download data into folder

from aisynbiopipeline.workflows.plasmidsaurus import get_access_token, download_results, get_credentials

CLIENT_ID = get_credentials("PLASMIDSAURUS_CLIENT_ID")
CLIENT_SECRET = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)
download_results(item_code, access_token, reception_dir)

In [ ]:
### TODO: How to ensure that archive was successfully unzipped?? Pipeline log file into home_dir?

In [ ]:
# Spot-check if duplicate read ID problem is fixed

plasmidsaurus_read_folder_name = item_code + '_reads'
os.listdir(os.path.join(reception_dir, plasmidsaurus_read_folder_name))

In [ ]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, '????')} | head

In [ ]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, '????')} | grep "????"


**Comment:**

- In this spot check, only found the read ID once in this file. I assume this means that we are not dealing with the same read duplication problem.

In [ ]:
from aisynbiopipeline.workflows.fastq_utils import create_manifest, parse_illumina_fastq_filename

folder = os.path.join(reception_dir, plasmidsaurus_read_folder_name)

manifest = create_manifest(folder, platform='plasmidsaurus_illumina')
manifest.head()




In [ ]:
len(manifest)

In [ ]:
# Plasmidsaurus provides a sort of sample manifest for download from the seqorder page. (not available with read download through API).
# Downloaded to my laptop, uploaded to home_dir, now copying to reception dir.

import shutil

shutil.copy2(os.path.join(home_dir, f'{item_code}-summary-report.csv'), reception_dir)

In [ ]:
# Create new seqorder folder in experimental_data/sequencing_data/ and libraries

seqorder = SeqOrder(seqorder_name, create=True)
short = Library(seqorder, 'Illumina', create=True)
# long = Library(seqorder, 'Nanopore', create=True)

In [ ]:
# Copy fastqs into respective library folder

from pathlib import Path
import shutil

reads_path = Path(os.path.join(reception_dir, f'{item_code}_reads'))

def rename_plasmidsaurus_read_file(file_name):
    aisynbio_filename = ('_').join(file_name.split('_')[2:])
    return aisynbio_filename

for index, row in manifest.iterrows():
    plasmidsaurus_basename = os.path.basename(row['fwd_fastq'])
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

for index, row in manifest.iterrows():
    plasmidsaurus_basename = os.path.basename(row['rvs_fastq'])
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

In [ ]:
aisynbio_basename

## Cross-checking this Plasmidsaurus order seqsamples in LIMS and creating SeqSamples.

In [ ]:
experiments = ['TFMN2', 'TFMN3', 'Gyorgy']

In [ ]:
short_manifest = short.create_manifest('received')

In [ ]:
short_manifest

In [ ]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)

%run util_simple.py

In [ ]:
# Cross-checking seq sample names

thisExpLIMSseqsamples_short = query_lims(
    'Seqsamples',
    filters={'Experiment': experiments}
)['Name'].to_list()

print(f"Are all short Plasmidsaurus seqsamples from order {item_code} in the LIMS?")
print(all([x in thisExpLIMSseqsamples_short for x in short_manifest['sample_name'].to_list()]))

In [ ]:
# Moving Gyorgy seqsamples into ~/storage/tmp/Gyorgy for zip and download to Box

import shutil

g_dir = '~/storage/tmp/Gyorgy'
os.makedirs(g_dir, exists_ok=True)

gyorgy_LIMS_seqsamples = query_lims(
    'Seqsamples',
    filters={'Experiment': 'Gyorgy'}
)['Name'].to_list()

gyorgy_seqsamples = [SeqSample(short, x) for x in gyorgy_LIMS_seqsamples]

for s in gyorgy_seqsamples:
    for i in s.path:
        shutil.move(i, g_dir)

archive_path = shutil.make_archive('Gyorgy', 'zip', root_dir=g_dir)
print(f"Created archive: {archive_path}")

In [ ]:
# TFMN2/3 seqsamples

LIMS_seqsamples = query_lims(
    'Seqsamples',
    filters={'Experiment': 'TFMN2', 'TFMN3'}
)['Sequencing sample', 'Experiment', 'Sample Name']

In [ ]:
measurements = LIMS_seqsamples.copy()
measurements['Type'] = 'Short_DNA_reads'
measurements['Protocol'] = pd.na
measurements['Who measured'] = 'Paul Hanke & technicians'
measurements['Lab'] = 'ANL & Plasmidsaurus'
measurements['Timestamp'] = ???

measurements.rename(columns={
    'Sequencing sample': 'Name',
    'Sample Name': 'Sample ID'
}, inplace=True)

measurements = measurements[['Name', 'Type', 'Experiment', 'Sample ID', 'Protocol', 'Who measured', 'Lab', 'Timestamp']]
measurements.to_csv(f'~/storage/tmp/{item_code}_measurements.csv')

In [ ]:
# Creating batch (list) of seqsamples for this seqorder

seqsamples = [SeqSample(short, row['Sequencing sample']) for _, row in LIMS_seqsamples.iterrows()]

## QA/QC

In [ ]:
# Create the trimmed subfolder
short.create_subfolder('trimmed')

In [ ]:
## fastp workers running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 1
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 2

In [ ]:
# Where should this code go?

from pathlib import Path

def get_fastp_params(library, seqsample):

    fwd_in_path = seqsample.received[0]
    fwd_in_file = os.path.basename(fwd_in_path)
    fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    fwd_out_path = os.path.join(library.path, 'trimmed', fwd_out_file)
    rvs_in_path = seqsample.received[1]
    rvs_in_file = os.path.basename(rvs_in_path)
    rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    rvs_out_path = os.path.join(library.path, 'trimmed', rvs_out_file)
    
    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    fastp_params = {
        'path_to_fwd': norm(fwd_in_path),
        'path_to_rev': norm(rvs_in_path),
        'path_to_fwd_out': norm(fwd_out_path),
        'path_to_rev_out': norm(rvs_out_path),
        'threads': 16,
        'polyG':5
    }

    return fastp_params

In [ ]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks:

results = []

for sample in seqsamples + gyorgy_seqsamples:
    result = client.send_task(
        'fastp.run',
        kwargs=get_fastp_params(short, sample),
        queue='fastp'
    )
    results.append(result)

In [ ]:
for i in results:
    print(i.status)

In [ ]:
all([(r.status=='SUCCESS') for r in results])

In [ ]:
from aisynbiopipeline.workflows.read_qc import run_multiqc
import shutil

multiqc_report = run_multiqc(short.path / 'trimmed')
multiqc_report_dir = os.path.dirname(multiqc_report)
multiqc_report_file = os.path.basename(multiqc_report)
dst_multiqc_report_file = os.path.join(home_dir, item_code + '_trimmed_' + multiqc_report_file)
shutil.copy(multiqc_report, dst_multiqc_report_file)

## Run Breseq

- no subsampling
- against ACN3500

In [ ]:
from aisynbiopipeline.workflows.reference_utils import list_reference_genomes

list_reference_genomes()

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.breseq_task 1
"""

In [ ]:
# Two breseq workers are running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 1
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 2

In [ ]:
# Create breseq dir

short.create_subfolder('breseq')

In [ ]:
short_manifest = short.create_manifest('received')
short_manifest

In [193]:
# Where should this code go?

# Specifies and assigns breseq parameters

from pathlib import Path

def define_breseq_params(seqsample, ref_filename, poly=True, fold_coverage=300, num_processors=4, polymorphism_frequency_cutoff=0.05):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': ref_filename,
        'polymorphism_prediction': poly,
        'limit_fold_coverage': fold_coverage,
        'num_processors': num_processors,
        'polymorphism_frequency_cutoff': polymorphism_frequency_cutoff
        
    }
    return breseq_params

In [ ]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks

results = []

for sample in seqsamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk', polymorphism_frequency_cutoff=0.005, fold_coverage=0),
        queue='breseq'
    )
    results.append(result)    

In [ ]:
sum([(r.status=='SUCCESS') for r in results])

In [ ]:
for i in results:
    print(i.status)
for i in results:
    print(i.result['output']) ## Check version_name for breseq run

In [ ]:
# Symlink to library breseq folder
output_symlinks_dir = os.path.join(breseq_folder, 'symlink_to_library_breseq_folder')
os.makedirs(output_symlinks_dir)

path_to_folder = short.path / 'breseq'
dst = os.path.join(output_symlinks_dir, 'breseq')
os.symlink(path_to_folder, dst)

## Breseq analysis TFMN2

- Create breseq objects for each seqsample and aggregate summary counts.
- Generate mutation table for all samples and write to csv for import into Google Sheets.

In [ ]:
exp = 'TFMN2'

In [ ]:
# Parent samples in this experiment

from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample

parentlib = Library(SeqOrder('Plasmidsaurus_9-13-2025_HTGS8F'), "Illumina")
parent_3500 = SeqSample(parentlib, 'ANLstock.ACN3500.colony2')
parent_3575 = SeqSample(parentlib, 'ANLstock.ACN3575.colony1')
parents = [parent_3500, parent_3575]

for p in parents:
    print(os.listdir(p.breseq))

In [ ]:
from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, genomic_region_from_features

genome3500 = os.path.join(get_ref_genomes_path(), 'ACN3500_NSS.gbk')

def get_region_parameter(genbank_file, feature_first, feature_last):
    genome, start, stop = genomic_region_from_features(genbank_file, feature_first, feature_last)
    region = genome + ":" + str(start) + "-" + str(stop)
    return region


In [68]:
# Where should this go?

def parse_seqsample_name(seqsample_name):
    
    import re

    pattern = re.compile(
        r'(TFMN2\.(ACN3500|ACN3575)\.(Iva|Mxb|Van)\.(DEL6kb|DEL6kb-DELvanK|DEL6kb-DELvanK-DELadeK|gDNA3560|gDNA3575|gDNA3749|gDNA3575-gDNA3749)\.([1-8]))\.T(\d{1,2})\.P$'
    )

    match = re.match(pattern, seqsample_name)

    if match:
        sample = str(match.group(1))
        strain = str(match.group(2))
        media = int(match.group(3))
        construct = int(match.group(4))
        replicate = str(match.group(5))
        transfer = int(match.group(6)) if isinstance(match.group(6), str) else None

    else:
        sample = 'NA'
        strain = 'NA'
        media = 'NA'
        construct = 'NA'
        replicate = 'NA'
        transfer = 'NA'

    return {'sample': sample, 'strain': strain, 'media': media, 'construct': construct, 'replicate': replicate, 'transfer': transfer}
        

In [ ]:
from aisynbiopipeline.workflows.breseq import Breseq
from aisynbiopipeline.workflows.mapping import PileupCol

def create_breseq_summary(seqsample_batch, version_name, output_path=None, regions=None, loci=None):
    
    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)
    
    rows = []
    
    for b in breseq_objects:
        
        row = {}
        
        try:
            b.count_reads()
            b.count_mutations()
            b.avg_coverage
            if regions:
                for key, value in regions.items():
                    b.get_region_average_coverage(value)
        except Exception as e:
            row.update({'seqsample': getattr(b, 'title', None)})
            row.update(parse_seqsample_name(getattr(b, 'title', None)))
            row.update({'error': str(e),
                        'input_read_count': None,
                        'used_read_count': None,
                        'mapped_read_count': None,
                        'consensus_mutation_count': None,
                        'polymorphism_mutation_count': None,
                        'average_cov': None,}
                        )
            if regions:
                for key, value in regions.items():
                    row.update({key: None})
            rows.append(row)
            print(f"Error loading Breseq from {b.title}: {e}")
            continue
    
        row['seqsample'] = getattr(b, 'title', None)
        row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row['error'] = None
        row['input_read_count'] = getattr(b, 'input_read_count', None)
        row['used_read_count'] = getattr(b, 'used_read_count', None)
        row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
        row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
        row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)
        row['average_cov'] = getattr(b, 'avg_coverage', None)
        
        if regions:
            for key, value in regions.items():
                row[key] = getattr(b, 'get_region_average_coverage', None)(value)
        
        if loci:
            for key, value in loci['mutations'].items():
                prefix = key
                basecalls = PileupCol(b.bam_path, value['locus']).basecalls
                locus_cov = len(basecalls)
                counts = Counter(basecalls)
                alt_freq = counts[value['alt_allele']] / locus_cov
                row[key+'_locus_cov'] = locus_cov
                row[key+'_alleleCounts'] = dict(counts)
                row[key+'_alt_allele_freq'] = alt_freq
        
        rows.append(row)
    
    breseq_summary = pd.DataFrame(rows)
    
    if regions:
        for key, value in regions.items():
            breseq_summary[key+'_CN'] = breseq_summary[key]/breseq_summary['average_cov']

    # Put rows and cols into correct order
    metadata_cols = ['seqsample', 'sample', 'strain', 'media', 'construct', 'replicate', 'transfer']
    other_cols = [col for col in breseq_summary.columns.to_list() if not in metadata_cols]
    breseq_summary = breseq_summary[metadata_cols + other_cols]
    breseq_summary.sort_values(['media', 'construct', 'strain', 'replicate', 'transfer'], inplace=True)

    # Write breseq run summary to csv
    if output_path:
        breseq_summary.to_csv(output_path)
    
    return breseq_summary

In [ ]:
# These strains are supposed to have the 6007 bp promoter deletion, including verR.
# Therefore, defining 'ver cassette' as verB through omega KmR cassette. 

regions = {
    # 'dgoA-Star': get_region_parameter(genome2821, 'dgoA-optimized-ADP1', 'dgoA-optimized-ADP1'),
    'ver_cassette': get_region_parameter(genome3500, 'omega KmR cassette', 'verB')
}

In [ ]:
loci = {
    'reference': 'ACN3500_NSS',
    'mutations': {
        'promoter_6kb.DEL': {
            'locus': (950040, 955945),  # added 50 bp from start and subtracted 50 bp from end to accomodate slightly different start and end loci between samples
            'ref_allele': '...',
            'alt_allele': ''
        },
        'vanK_1-bp.DEL': {
            'locus': (974618, 974618),
            'ref_allele': 'G',
            'alt_allele': ''
        },
        'ACIAD_RS08195<->fecI.SNP': {
            'locus': (1405240, 1405241),
            'ref_allele': 'T',
            'alt_allele': 'A'
        },
        'dsbD.Q75stop': {
            'locus': (3465740, 3465740),
            'ref_allele': 'C',
            'alt_allele': 'T'
        },
        'iscR_2-bp.DEL': {
            'locus': (1405240, 1405241),
            'ref_allele': 'GA',
            'alt_allele': ''
        },
        'rpoD_3-bp.DEL': {
            'locus': (2860637, 2860639),
            'ref_allele': 'GAA',
            'alt_allele': ''
        },
        'adeK_2-bp.DEL': {
            'locus': (2883783, 2883784),
            'ref_allele': 'GA',
            'alt_allele': ''
        },
        'mnmA.R245H': {
            'locus': (1230459, 1230459),
            'ref_allele': 'C',
            'alt_allele': 'T'
        },
        'ACIAD_RS01630_2-bp.DEL': {
            'locus': (345201, 345202),
            'ref_allele': 'AA',
            'alt_allele': ''
        }
    }
}

In [ ]:
regions_to_include = ['ver_cassette']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

seqsample_batch = parents + seqsamples 
breseq_version_name = ????

breseq_folder = home_dir + '/' + exp + '/' + breseq_version_name
os.makedirs(breseq_folder, exist_ok=True)

breseq_summary = create_breseq_summary(
    seqsample_batch,
    breseq_version_name,
    output_path=os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_summary.csv'),
    regions=regions_sub,
    loci=loci
)

# create_html_comparison(
#     seqsample_batch,
#     breseq_version_name,
#     os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_comparison.html')
# )

In [ ]:
# Missing: parse, reformat, curate, save breseq comparison

......

comparison_path = os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_comparison.csv')

In [ ]:
# Create mutation comparison files

from aisynbiopipeline.workflows.breseq import compare_gdiff

breseq_objects = []
for s in seqsample_batch:
    breseq_folder = s.library.path / 'breseq' / s.sample_name/ breseq_version_name
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

reference = 'ACN3500_NSS.gbk'
gdiffs = [b.gd_file for b in breseq_objects]

table_format = 'html'
outfile = os.path.join(breseq_folder, f'{exp}_{breseq_version_name}.{table_format}')
html = compare_gdiff(reference, outfile, gdiffs, format=table_format)

table_format = 'csv'
outfile = os.path.join(breseq_folder, f'{exp}_{breseq_version_name}.{table_format}')
csv = compare_gdiff(reference, outfile, gdiffs, format=table_format)

In [ ]:
compare_df = pd.read_csv(csv)

All of the following transformations should possibly go into a different script...

In [ ]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

cols_with_diff_vals_for_same_mutation = ['new_read_count', 'new_read_count_basis', 'ref_read_count',
     'ref_read_count_basis', 'multiple_polymorphic_SNPs_in_same_codon',
     'repeat_new_copies', 'repeat_ref_copies'
    ]
cols_uninformative = ['clone', 'mutator_status', 'population', 'time', 'treatment']
cols_to_keep = [col for col in compare_df.columns.to_list() if (col not in cols_with_diff_vals_for_same_mutation) and (col not in cols_uninformative)]

compare_df = compare_df[cols_to_keep]

In [ ]:
compare_df.head()

In [ ]:
index = [col for col in compare_df.columns if (col!='title') and (col!='frequency')]

df = compare_df.pivot(index=index, columns = 'title', values = 'frequency').sort_index(level='position')
df = df.fillna(0)
# df = df.loc[~(df > 0).all(axis=1)] # Only mutations that don't appear in all samples
df = df[breseq_summary['seqsample'].to_list()]  # order the comparison just like the summary

In [ ]:
df_greater5 = df.loc[(df > 0.05).any(axis=1)].dropna(how="all")
df_greater80 = df.loc[(df > 0.80).any(axis=1)].dropna(how="all")

In [ ]:
# Write reformatted mutations to csv.

df.to_csv(os.path.join(breseq_folder, f'mutation_comparison_{item_code}_{breseq_version_name}_reformatted.csv'))
df_greater5.to_csv(os.path.join(breseq_folder, f'mutation_comparison_{item_code}_{breseq_version_name}_reformatted_greater5.csv'))
df_greater80.to_csv(os.path.join(breseq_folder, f'mutation_comparison_{item_code}_{breseq_version_name}_reformatted_greater80.csv'))

## Breseq analysis TFMN3

- Create breseq objects for each seqsample and aggregate summary counts.
- Generate mutation table for all samples and write to csv for import into Google Sheets.

In [ ]:
exp = 'TFMN3'

In [ ]:
# Parent samples in this experiment

from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample

parentlib = Library(SeqOrder('Plasmidsaurus_9-13-2025_HTGS8F'), "Illumina")
parent_3500 = SeqSample(parentlib, 'ANLstock.ACN3500.colony2')
parent_3575 = SeqSample(parentlib, 'ANLstock.ACN3575.colony1')

parentlib = Library(SeqOrder('Plasmidsaurus_2026-02-13_QYYTMB'), "Illumina")
parent_3749 = SeqSample(parentlib, 'ANLstock.ACN3749.colony1')

parents = [parent_3500, parent_3575, parent_3749]

for p in parents:
    print(os.listdir(p.breseq))

In [ ]:
from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, genomic_region_from_features

genome3500 = os.path.join(get_ref_genomes_path(), 'ACN3500_NSS.gbk')

def get_region_parameter(genbank_file, feature_first, feature_last):
    genome, start, stop = genomic_region_from_features(genbank_file, feature_first, feature_last)
    region = genome + ":" + str(start) + "-" + str(stop)
    return region


In [68]:
# Where should this go?

def parse_seqsample_name(seqsample_name):
    
    import re

    pattern = re.compile(
        r'(TFMN3\.(ACN3500|ACN3575|ACN3749)\.(Mxb)\.(noDNA)\.([2-5]))\.T(\d{1,2})\.P$'
    )

    match = re.match(pattern, seqsample_name)

    if match:
        sample = str(match.group(1))
        strain = str(match.group(2))
        media = int(match.group(3))
        construct = int(match.group(4))
        replicate = str(match.group(5))
        transfer = int(match.group(6)) if isinstance(match.group(6), str) else None

    else:
        sample = 'NA'
        strain = 'NA'
        media = 'NA'
        construct = 'NA'
        replicate = 'NA'
        transfer = 'NA'

    return {'sample': sample, 'strain': strain, 'media': media, 'construct': construct, 'replicate': replicate, 'transfer': transfer}
        

In [ ]:
from aisynbiopipeline.workflows.breseq import Breseq
from aisynbiopipeline.workflows.mapping import PileupCol

def create_breseq_summary(seqsample_batch, version_name, output_path=None, regions=None, loci=None):
    
    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)
    
    rows = []
    
    for b in breseq_objects:
        
        row = {}
        
        try:
            b.count_reads()
            b.count_mutations()
            b.avg_coverage
            if regions:
                for key, value in regions.items():
                    b.get_region_average_coverage(value)
        except Exception as e:
            row.update({'seqsample': getattr(b, 'title', None)})
            row.update(parse_seqsample_name(getattr(b, 'title', None)))
            row.update({'error': str(e),
                        'input_read_count': None,
                        'used_read_count': None,
                        'mapped_read_count': None,
                        'consensus_mutation_count': None,
                        'polymorphism_mutation_count': None,
                        'average_cov': None,}
                        )
            if regions:
                for key, value in regions.items():
                    row.update({key: None})
            rows.append(row)
            print(f"Error loading Breseq from {b.title}: {e}")
            continue
    
        row['seqsample'] = getattr(b, 'title', None)
        row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row['error'] = None
        row['input_read_count'] = getattr(b, 'input_read_count', None)
        row['used_read_count'] = getattr(b, 'used_read_count', None)
        row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
        row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
        row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)
        row['average_cov'] = getattr(b, 'avg_coverage', None)
        
        if regions:
            for key, value in regions.items():
                row[key] = getattr(b, 'get_region_average_coverage', None)(value)
        
        if loci:
            for key, value in loci['mutations'].items():
                prefix = key
                basecalls = PileupCol(b.bam_path, value['locus']).basecalls
                locus_cov = len(basecalls)
                counts = Counter(basecalls)
                alt_freq = counts[value['alt_allele']] / locus_cov
                row[key+'_locus_cov'] = locus_cov
                row[key+'_alleleCounts'] = dict(counts)
                row[key+'_alt_allele_freq'] = alt_freq
        
        rows.append(row)
    
    breseq_summary = pd.DataFrame(rows)
    
    if regions:
        for key, value in regions.items():
            breseq_summary[key+'_CN'] = breseq_summary[key]/breseq_summary['average_cov']

    # Put rows and cols into correct order
    metadata_cols = ['seqsample', 'sample', 'strain', 'media', 'construct', 'replicate', 'transfer']
    other_cols = [col for col in breseq_summary.columns.to_list() if not in metadata_cols]
    breseq_summary = breseq_summary[metadata_cols + other_cols]
    breseq_summary.sort_values(['media', 'construct', 'strain', 'replicate', 'transfer'], inplace=True)

    # Write breseq run summary to csv
    if output_path:
        breseq_summary.to_csv(output_path)
    
    return breseq_summary

In [ ]:
# These strains are supposed to have the 6007 bp promoter deletion, including verR.
# Therefore, defining 'ver cassette' as verB through omega KmR cassette. 

regions = {
    # 'dgoA-Star': get_region_parameter(genome2821, 'dgoA-optimized-ADP1', 'dgoA-optimized-ADP1'),
    'ver_cassette': get_region_parameter(genome3500, 'omega KmR cassette', 'verB')
}

In [ ]:
loci = {
    'reference': 'ACN3500_NSS',
    'mutations': {
        'promoter_6kb.DEL': {
            'locus': (950040, 955945),  # added 50 bp from start and subtracted 50 bp from end to accomodate slightly different start and end loci between samples
            'ref_allele': '...',
            'alt_allele': ''
        },
        'vanK_1-bp.DEL': {
            'locus': (974618, 974618),
            'ref_allele': 'G',
            'alt_allele': ''
        },
        'ACIAD_RS08195<->fecI.SNP': {
            'locus': (1405240, 1405241),
            'ref_allele': 'T',
            'alt_allele': 'A'
        },
        'dsbD.Q75stop': {
            'locus': (3465740, 3465740),
            'ref_allele': 'C',
            'alt_allele': 'T'
        },
        'iscR_2-bp.DEL': {
            'locus': (1405240, 1405241),
            'ref_allele': 'GA',
            'alt_allele': ''
        },
        'rpoD_3-bp.DEL': {
            'locus': (2860637, 2860639),
            'ref_allele': 'GAA',
            'alt_allele': ''
        },
        'adeK_2-bp.DEL': {
            'locus': (2883783, 2883784),
            'ref_allele': 'GA',
            'alt_allele': ''
        },
        'mnmA.R245H': {
            'locus': (1230459, 1230459),
            'ref_allele': 'C',
            'alt_allele': 'T'
        },
        'ACIAD_RS01630_2-bp.DEL': {
            'locus': (345201, 345202),
            'ref_allele': 'AA',
            'alt_allele': ''
        }
    }
}

In [ ]:
regions_to_include = ['ver_cassette']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

seqsample_batch = parents + seqsamples 
breseq_version_name = ????

breseq_folder = home_dir + '/' + exp + '/' + breseq_version_name
os.makedirs(breseq_folder, exist_ok=True)

breseq_summary = create_breseq_summary(
    seqsample_batch,
    breseq_version_name,
    output_path=os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_summary.csv'),
    regions=regions_sub,
    loci=loci
)

# create_html_comparison(
#     seqsample_batch,
#     breseq_version_name,
#     os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_comparison.html')
# )

In [ ]:
# Missing: parse, reformat, curate, save breseq comparison

......

comparison_path = os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_comparison.csv')

In [ ]:
# Create mutation comparison files

from aisynbiopipeline.workflows.breseq import compare_gdiff

breseq_objects = []
for s in seqsample_batch:
    breseq_folder = s.library.path / 'breseq' / s.sample_name/ breseq_version_name
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

reference = 'ACN3500_NSS.gbk'
gdiffs = [b.gd_file for b in breseq_objects]

table_format = 'html'
outfile = os.path.join(breseq_folder, f'{exp}_{breseq_version_name}.{table_format}')
html = compare_gdiff(reference, outfile, gdiffs, format=table_format)

table_format = 'csv'
outfile = os.path.join(breseq_folder, f'{exp}_{breseq_version_name}.{table_format}')
csv = compare_gdiff(reference, outfile, gdiffs, format=table_format)

In [ ]:
compare_df = pd.read_csv(csv)

All of the following transformations should possibly go into a different script...

In [ ]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

cols_with_diff_vals_for_same_mutation = ['new_read_count', 'new_read_count_basis', 'ref_read_count',
     'ref_read_count_basis', 'multiple_polymorphic_SNPs_in_same_codon',
     'repeat_new_copies', 'repeat_ref_copies'
    ]
cols_uninformative = ['clone', 'mutator_status', 'population', 'time', 'treatment']
cols_to_keep = [col for col in compare_df.columns.to_list() if (col not in cols_with_diff_vals_for_same_mutation) and (col not in cols_uninformative)]

compare_df = compare_df[cols_to_keep]

In [ ]:
compare_df.head()

In [ ]:
index = [col for col in compare_df.columns if (col!='title') and (col!='frequency')]

df = compare_df.pivot(index=index, columns = 'title', values = 'frequency').sort_index(level='position')
df = df.fillna(0)
# df = df.loc[~(df > 0).all(axis=1)] # Only mutations that don't appear in all samples
df = df[breseq_summary['seqsample'].to_list()]  # order the comparison just like the summary

In [ ]:
df_greater5 = df.loc[(df > 0.05).any(axis=1)].dropna(how="all")
df_greater80 = df.loc[(df > 0.80).any(axis=1)].dropna(how="all")

In [ ]:
# Write reformatted mutations to csv.

df.to_csv(os.path.join(breseq_folder, f'mutation_comparison_{item_code}_{breseq_version_name}_reformatted.csv'))
df_greater5.to_csv(os.path.join(breseq_folder, f'mutation_comparison_{item_code}_{breseq_version_name}_reformatted_greater5.csv'))
df_greater80.to_csv(os.path.join(breseq_folder, f'mutation_comparison_{item_code}_{breseq_version_name}_reformatted_greater80.csv'))